# Hunyuan3D-2 MV: Colab L4 GPU Headless High-Res & Warm-Batch Pipeline

支援高解析度幾何提取 (Octree 384, 50 steps, 40k faces)、精準外部 Flood-Fill 去背保護（防止白肚皮破洞與刺間灰邊），以及單一 Session 多模型熱啟動批次推論。

In [ ]:
import os, shutil, subprocess, sys, re
from pathlib import Path

def run_cmd(cmd, cwd=None):
    res = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"❌ 指令失敗: {' '.join(cmd)}", flush=True)
        print("--- STDERR ---", flush=True)
        print(res.stderr[-3000:], flush=True)
        print("--- STDOUT ---", flush=True)
        print(res.stdout[-3000:], flush=True)
        raise RuntimeError(f"執行失敗: {res.stderr[-500:]}")
    return res.stdout

SETUP_MARKER = Path("/content/.hy3d_setup_done_v2")
COMFY_DIR = Path("/content/ComfyUI")

# 1. 確保 ComfyUI 本體完整就緒
if not (COMFY_DIR / "main.py").exists():
    print("正在下載 ComfyUI 核心庫...")
    if COMFY_DIR.exists():
        shutil.rmtree(str(COMFY_DIR), ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)])

# 2. 確保 custom_nodes 目錄與自訂節點完整就緒
custom_nodes = COMFY_DIR / "custom_nodes"
custom_nodes.mkdir(parents=True, exist_ok=True)

hy3d_dir = custom_nodes / "ComfyUI-Hunyuan3DWrapper"
essentials_dir = custom_nodes / "ComfyUI_essentials"

if not (hy3d_dir / "requirements.txt").exists():
    print("下載 ComfyUI-Hunyuan3DWrapper...")
    if hy3d_dir.exists():
        shutil.rmtree(str(hy3d_dir), ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/kijai/ComfyUI-Hunyuan3DWrapper.git", str(hy3d_dir)])

if not essentials_dir.exists():
    print("下載 ComfyUI_essentials...")
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/cubiq/ComfyUI_essentials.git", str(essentials_dir)])

# 3. 安裝依賴與注入高精度補丁
if not SETUP_MARKER.exists():
    print("安裝 Python 模組依賴...", flush=True)
    run_cmd([sys.executable, "-m", "pip", "install", "--break-system-packages", "-r", str(COMFY_DIR / "requirements.txt")])
    run_cmd([sys.executable, "-m", "pip", "install", "--break-system-packages", "-r", str(hy3d_dir / "requirements.txt")])
    run_cmd([sys.executable, "-m", "pip", "install", "--break-system-packages", "transformers==4.46.0", "accelerate==1.1.1", "diffusers==0.30.0", "rembg[gpu]", "trimesh", "pymeshlab", "xatlas", "huggingface_hub", "scipy"])

    # 補丁 1：確保 pipelines.py 支援 DINOv2 權重鍵名映射且清空所有 meta tensor
    pipe_py = hy3d_dir / "hy3dgen" / "shapegen" / "pipelines.py"
    if pipe_py.exists():
        pipe_code = pipe_py.read_text(encoding="utf-8")
        if "🔥 Conditioner 權重加載" not in pipe_code:
            cond_replacement = '''        if \x27conditioner\x27 in new_sd:
            c_sd = new_sd[\x27conditioner\x27]
            loaded_count = 0
            cond_params = list(conditioner.named_parameters())
            for name, param in cond_params:
                val = c_sd.get(name)
                if val is None:
                    alias = name.replace(\x27.attention.q_proj.\x27, \x27.attention.attention.query.\x27) \\
                                .replace(\x27.attention.k_proj.\x27, \x27.attention.attention.key.\x27) \\
                                .replace(\x27.attention.v_proj.\x27, \x27.attention.attention.value.\x27) \\
                                .replace(\x27.attention.o_proj.\x27, \x27.attention.output.dense.\x27)
                    val = c_sd.get(alias)
                if val is None:
                    alias2 = name.replace(\x27.attention.attention.query.\x27, \x27.attention.q_proj.\x27) \\
                                 .replace(\x27.attention.attention.key.\x27, \x27.attention.k_proj.\x27) \\
                                 .replace(\x27.attention.attention.value.\x27, \x27.attention.v_proj.\x27) \\
                                 .replace(\x27.attention.output.dense.\x27, \x27.attention.o_proj.\x27)
                    val = c_sd.get(alias2)
                if val is not None:
                    set_module_tensor_to_device(conditioner, name, device=offload_device, dtype=dtype, value=val)
                    loaded_count += 1
                else:
                    set_module_tensor_to_device(conditioner, name, device=offload_device, dtype=dtype, value=torch.zeros(param.shape, dtype=dtype))
            print(f"🔥 Conditioner 權重加載: 成功加載 {loaded_count}/{len(cond_params)} 個參數！", flush=True)

        for m in [conditioner, model, vae]:
            for p_name, param in m.named_parameters():
                if getattr(param, \x27is_meta\x27, False):
                    set_module_tensor_to_device(m, p_name, device=offload_device, dtype=dtype, value=torch.zeros(param.shape, dtype=dtype))
            for b_name, buf in m.named_buffers():
                if getattr(buf, \x27is_meta\x27, False):
                    set_module_tensor_to_device(m, b_name, device=offload_device, dtype=buf.dtype, value=torch.zeros(buf.shape, dtype=buf.dtype))

        image_processor = instantiate_from_config(config[\x27image_processor\x27])'''

            cond_regex = re.compile(r"        if \x27conditioner\x27 in new_sd:.*?image_processor = instantiate_from_config\(config\[\x27image_processor\x27\]\)", re.DOTALL)
            if cond_regex.search(pipe_code):
                pipe_code = cond_regex.sub(cond_replacement, pipe_code, count=1)
                pipe_py.write_text(pipe_code, encoding="utf-8")
                print("✅ 已對 pipelines.py 注入完整 DINOv2 權重映射與 Meta Tensor 清理補丁！")

    # 補丁 2：升級 ComfyUI_essentials ImageRemoveBackground+ 為「外部純白 Flood-Fill 精準輪廓去背」
    # 徹底杜絕 u2net 誤將企鵝側面雪白肚皮挖洞，並消除刺蝟尖刺間的半透明灰邊
    ess_img_py = essentials_dir / "image.py"
    if ess_img_py.exists():
        e_code = ess_img_py.read_text(encoding="utf-8")
        if "EXTERIOR_FLOOD_FILL_ALPHA" not in e_code:
            target_fn = "def execute(self, rembg_session, image):"
            replacement_fn = '''def execute(self, rembg_session, image):
        # EXTERIOR_FLOOD_FILL_ALPHA: 針對純白背景 (#FFFFFF) 角色圖進行無損外部 Flood-Fill 去背
        import numpy as np
        from collections import deque
        B, H, W, C = image.shape
        out_imgs = []
        out_masks = []
        for b in range(B):
            rgb = image[b, :, :, :3].cpu().numpy()
            luma = 0.299 * rgb[:, :, 0] + 0.587 * rgb[:, :, 1] + 0.114 * rgb[:, :, 2]
            sat = rgb.max(axis=2) - rgb.min(axis=2)
            corners_white = (luma[0,0] > 0.95 and luma[0,-1] > 0.95 and luma[-1,0] > 0.95 and luma[-1,-1] > 0.95)
            if corners_white:
                is_white = (luma > 0.968) & (sat < 0.032)
                ext = np.zeros((H, W), dtype=bool)
                q = deque()
                for x in range(W):
                    if is_white[0, x]: ext[0, x] = True; q.append((0, x))
                    if is_white[H-1, x] and not ext[H-1, x]: ext[H-1, x] = True; q.append((H-1, x))
                for y in range(H):
                    if is_white[y, 0] and not ext[y, 0]: ext[y, 0] = True; q.append((y, 0))
                    if is_white[y, W-1] and not ext[y, W-1]: ext[y, W-1] = True; q.append((y, W-1))
                while q:
                    cy, cx = q.popleft()
                    for ny, nx in ((cy-1,cx), (cy+1,cx), (cy,cx-1), (cy,cx+1)):
                        if 0 <= ny < H and 0 <= nx < W and is_white[ny, nx] and not ext[ny, nx]:
                            ext[ny, nx] = True
                            q.append((ny, nx))
                alpha = (~ext).astype(np.float32)
                from scipy.ndimage import binary_erosion, gaussian_filter
                core = binary_erosion(~ext, iterations=1)
                smooth_a = gaussian_filter(alpha, sigma=0.75)
                alpha = np.where(core, 1.0, np.where(ext, 0.0, smooth_a))
                a_tensor = torch.from_numpy(alpha).to(image.device, dtype=image.dtype).unsqueeze(-1)
                rgba = torch.cat([image[b, :, :, :3], a_tensor], dim=-1)
                out_imgs.append(rgba)
                out_masks.append(a_tensor.squeeze(-1))
                print(f"✨ [ExteriorFloodFill] 精準去背完成 (前景佔比: {alpha.mean()*100:.2f}%)", flush=True)
        if len(out_imgs) == B:
            return (torch.stack(out_imgs, dim=0), torch.stack(out_masks, dim=0))'''
            if target_fn in e_code:
                e_code = e_code.replace(target_fn, replacement_fn, 1)
                ess_img_py.write_text(e_code, encoding="utf-8")
                print("✅ 已對 ComfyUI_essentials 注入 Exterior Flood-Fill 精準去背補丁！")

    SETUP_MARKER.touch()
    print("✅ 核心軟體環境已安裝就緒！")
else:
    print("✅ 重用已配置的 Hunyuan3D-2 軟體環境 (Warm Session)。")

# 4. 下載 Hunyuan3D-2 MV 核心 DiT 權重 (約 4.9 GB)
diff_dir = COMFY_DIR / "models" / "diffusion_models"
diff_dir.mkdir(parents=True, exist_ok=True)
model_file = diff_dir / "hunyuan3d-dit-v2-mv-fp16.safetensors"

if not model_file.exists() or model_file.stat().st_size < 4 * 1024 * 1024 * 1024:
    print("正在自 HuggingFace 下載 hunyuan3d-dit-v2-mv-fp16.safetensors (約 4.9GB)...", flush=True)
    from huggingface_hub import hf_hub_download
    dl_path = hf_hub_download(
        repo_id="tencent/Hunyuan3D-2mv",
        filename="hunyuan3d-dit-v2-mv/model.fp16.safetensors",
        local_dir="/content/hf_temp"
    )
    shutil.move(dl_path, str(model_file))
    print(f"✅ 模型權重下載成功！大小: {model_file.stat().st_size / (1024*1024):.2f} MB")
else:
    print(f"✅ 核心權重已存在 ({model_file.stat().st_size / (1024*1024):.2f} MB)，跳過下載。")


In [ ]:
import json, time, requests, subprocess, sys
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
PORT = 8188
BASE_URL = f"http://127.0.0.1:{PORT}"
LOG_PATH = Path("/content/comfyui.log")

try:
    resp = requests.get(f"{BASE_URL}/system_stats", timeout=2)
    resp.raise_for_status()
    print("🔥 使用背景現存的熱啟動 ComfyUI 服務 (VRAM 已常駐模型)")
except Exception:
    print("正在啟動 ComfyUI 服務...")
    log_f = open(LOG_PATH, "a")
    cmd = [sys.executable, str(COMFY_DIR / "main.py"), "--listen", "127.0.0.1", "--port", str(PORT), "--highvram", "--disable-auto-launch"]
    proc = subprocess.Popen(cmd, cwd=str(COMFY_DIR), stdout=log_f, stderr=subprocess.STDOUT)
    for _ in range(120):
        if proc.poll() is not None:
            raise RuntimeError(f"ComfyUI 啟動失敗；查看 {LOG_PATH}\n" + LOG_PATH.read_text()[-3000:])
        try:
            requests.get(f"{BASE_URL}/system_stats", timeout=2).raise_for_status()
            print("✅ ComfyUI 啟動就緒！")
            break
        except Exception:
            time.sleep(2)
    else:
        raise TimeoutError(f"ComfyUI 啟動逾時；查看 {LOG_PATH}")

In [ ]:
import os, shutil, json, time, uuid, requests
from pathlib import Path

PORT = 8188
BASE_URL = f"http://127.0.0.1:{PORT}"
LOG_PATH = Path("/content/comfyui.log")
COMFY_INPUT = Path("/content/ComfyUI/input")
COMFY_OUTPUT = Path("/content/ComfyUI/output")
COMFY_INPUT.mkdir(parents=True, exist_ok=True)
COMFY_OUTPUT.mkdir(parents=True, exist_ok=True)

workflow_file = Path("/content/workflow_api.json")
assert workflow_file.exists(), f"找不到工作流檔: {workflow_file}"
base_workflow = json.loads(workflow_file.read_text(encoding="utf-8"))

# 支援單一任務或批次佇列 (HY3D_BATCH_JSON)
batch_json_path = os.environ.get("HY3D_BATCH_JSON", "")
if batch_json_path and os.path.exists(batch_json_path):
    jobs = json.loads(Path(batch_json_path).read_text(encoding="utf-8"))
else:
    jobs = [{
        "name": "single_job",
        "front": os.environ.get("HY3D_FRONT_IMAGE", "/content/input_front.png"),
        "back": os.environ.get("HY3D_BACK_IMAGE", "/content/input_back.png"),
        "side": os.environ.get("HY3D_SIDE_IMAGE", "/content/input_side.png"),
        "faces": int(os.environ.get("HY3D_TARGET_FACES", "40000")),
        "output": os.environ.get("HY3D_OUTPUT_PATH", "/content/output_model.glb"),
    }]

print(f"📋 準備執行 {len(jobs)} 個 3D 模型生成任務...")

for idx, job in enumerate(jobs, 1):
    j_name = job.get("name", f"model_{idx}")
    front_img = job["front"]
    back_img = job["back"]
    side_img = job["side"]
    target_faces = int(job.get("faces", 40000))
    output_path = Path(job["output"])
    output_path.parent.mkdir(parents=True, exist_ok=True)

    assert os.path.exists(front_img), f"找不到正面圖片: {front_img}"
    assert os.path.exists(back_img), f"找不到背面圖片: {back_img}"
    assert os.path.exists(side_img), f"找不到側面圖片: {side_img}"

    job_id = f"{j_name}_{uuid.uuid4().hex[:6]}"
    fname_front = f"front_{job_id}.png"
    fname_back = f"back_{job_id}.png"
    fname_side = f"side_{job_id}.png"

    shutil.copy2(front_img, COMFY_INPUT / fname_front)
    shutil.copy2(back_img, COMFY_INPUT / fname_back)
    shutil.copy2(side_img, COMFY_INPUT / fname_side)

    prefix = f"hy3d_out_{job_id}"
    workflow = json.loads(json.dumps(base_workflow))
    workflow["10"]["inputs"]["image"] = fname_front
    workflow["11"]["inputs"]["image"] = fname_back
    workflow["12"]["inputs"]["image"] = fname_side
    workflow["32"]["inputs"]["max_facenum"] = target_faces
    workflow["40"]["inputs"]["filename_prefix"] = prefix

    t_start = time.time()
    client_id = str(uuid.uuid4())
    prompt_res = requests.post(f"{BASE_URL}/prompt", json={"prompt": workflow, "client_id": client_id}, timeout=30).json()
    prompt_id = prompt_res.get("prompt_id")
    if not prompt_id:
        raise RuntimeError(f"ComfyUI 提交任務失敗: {json.dumps(prompt_res, indent=2)}")
    print(f"\n🚀 [{idx}/{len(jobs)}] 啟動推論任務 [{j_name}] (Prompt ID: {prompt_id}, 目標面數: {target_faces})...", flush=True)

    last_hist = None
    while time.time() - t_start < 1800:
        try:
            hist_resp = requests.get(f"{BASE_URL}/history/{prompt_id}", timeout=60)
            if hist_resp.status_code == 200:
                data = hist_resp.json()
                if prompt_id in data:
                    last_hist = data[prompt_id]
                    break
        except requests.exceptions.RequestException as req_err:
            print(f"⏳ [{j_name}] 運算中... ({type(req_err).__name__})", flush=True)
        time.sleep(3)

    if not last_hist:
        raise TimeoutError(f"[{j_name}] 推論執行逾時")

    status_info = last_hist.get("status", {})
    if not status_info.get("completed", False) or status_info.get("status_str") == "error":
        if LOG_PATH.exists():
            print(LOG_PATH.read_text()[-4000:])
        raise RuntimeError(f"[{j_name}] ComfyUI 推論報錯: {status_info.get('messages')}")

    found_glb = None
    for p in COMFY_OUTPUT.rglob("*.glb"):
        if prefix in p.name:
            found_glb = p
            break
    if not found_glb:
        glbs = sorted(list(COMFY_OUTPUT.rglob("*.glb")), key=os.path.getmtime, reverse=True)
        if glbs:
            found_glb = glbs[0]

    assert found_glb and found_glb.exists(), f"[{j_name}] 未找到產出的 .glb 檔案！"
    shutil.copy2(found_glb, output_path)
    elapsed = time.time() - t_start
    mode_tag = "冷啟動首模型 (含模型載入 VRAM)" if idx == 1 else "🔥 熱啟動 (VRAM 常駐)"
    print(f"✅ [{idx}/{len(jobs)}] [{j_name}] 生成完成！({mode_tag}) 耗時: {elapsed:.2f} 秒 | 大小: {output_path.stat().st_size / (1024*1024):.2f} MB -> {output_path}", flush=True)
